In [3]:
import numpy as np

def pagerank(adjacency_list, num_nodes, damping=0.85, max_iter=100, tol=1e-6):
    """
    Вычисляет PageRank для заданного ориентированного графа.

    Параметры:
        adjacency_list (dict): Словарь, где ключ — id узла, значение — список соседей, в которые ведут исходящие рёбра.
        num_nodes (int): Общее количество узлов (должны быть пронумерованы от 0 до num_nodes - 1).
        damping (float): Коэффициент затухания (обычно 0.85).
        max_iter (int): Максимальное число итераций.
        tol (float): Порог сходимости (по L1-норме разности векторов).

    Возвращает:
        np.ndarray: Вектор PageRank размера num_nodes.
    """
    # Инициализация PageRank: равномерное распределение
    pr = np.full(num_nodes, 1.0 / num_nodes)

    # Подсчёт исходящих степеней
    out_degree = np.zeros(num_nodes, dtype=int)
    for node in range(num_nodes):
        out_degree[node] = len(adjacency_list.get(node, []))

    for _ in range(max_iter):
        new_pr = np.full(num_nodes, (1 - damping) / num_nodes)

        for node in range(num_nodes):
            if out_degree[node] == 0:
                # Узел без исходящих рёбер — "утечка" массы (random jump)
                continue
            share = pr[node] / out_degree[node]
            for neighbor in adjacency_list.get(node, []):
                if 0 <= neighbor < num_nodes:
                    new_pr[neighbor] += damping * share

        # Проверка сходимости
        if np.sum(np.abs(new_pr - pr)) < tol:
            break
        pr = new_pr

    return pr


# Пример использования
if __name__ == "__main__":
    # Пример графа: adjacency_list[i] — список узлов, в которые ведут рёбра из i
    graph = {
        1: [2],
        2: [1, 3],
        3: [1, 4],
        4: [3, 2]
    }
    N = 4  # количество узлов (0..3)

    ranks = pagerank(graph, N)
    for i, r in enumerate(ranks):
        print(f"PageRank узла {i}: {r:.6f}")

PageRank узла 0: 0.037500
PageRank узла 1: 0.156937
PageRank узла 2: 0.170897
PageRank узла 3: 0.110131


In [4]:
import numpy as np

def pagerank_from_1(adjacency_list, num_nodes, damping=0.65, max_iter=100, tol=1e-6):
    """
    Вычисляет PageRank для ориентированного графа с нумерацией вершин от 1 до num_nodes.

    Параметры:
        adjacency_list (dict): Ключи — номера вершин от 1 до num_nodes,
                               значения — списки целевых вершин (тоже от 1 до num_nodes).
        num_nodes (int): Общее количество вершин (нумерация с 1).
        damping (float): Коэффициент затухания (по умолчанию 0.85).
        max_iter (int): Максимальное число итераций.
        tol (float): Порог сходимости (L1-норма разности векторов).

    Возвращает:
        dict: Словарь {вершина (int): PageRank (float)}, вершины от 1 до num_nodes.
    """
    # Инициализация: равномерное распределение
    pr = np.full(num_nodes, 1.0 / num_nodes)

    # Подготовка данных: преобразуем в индексацию с 0
    adj_0indexed = {}
    out_degree = np.zeros(num_nodes, dtype=int)

    for node in range(1, num_nodes + 1):
        neighbors = adjacency_list.get(node, [])
        # Преобразуем соседей в 0-индексацию
        neighbors_0 = [n - 1 for n in neighbors if 1 <= n <= num_nodes]
        adj_0indexed[node - 1] = neighbors_0
        out_degree[node - 1] = len(neighbors_0)

    # Итерации PageRank
    for _ in range(max_iter):
        new_pr = np.full(num_nodes, (1 - damping) / num_nodes)

        for node in range(num_nodes):
            if out_degree[node] == 0:
                continue  # "висячий" узел — не передаёт вес
            share = pr[node] / out_degree[node]
            for neighbor in adj_0indexed[node]:
                new_pr[neighbor] += damping * share

        if np.sum(np.abs(new_pr - pr)) < tol:
            pr = new_pr
            break
        pr = new_pr

    # Возврат результата с нумерацией от 1
    return {i + 1: pr[i] for i in range(num_nodes)}


# Пример использования
if __name__ == "__main__":
    # Пример графа: вершины от 1 до 4
    graph = {
        1: [2],
        2: [1, 3],
        3: [1, 4],
        4: [3, 2]
    }
    N = 4  # вершины: 1, 2, 3, 4

    ranks = pagerank_from_1(graph, N)

    for node in sorted(ranks):
        print(f"PageRank вершины {node}: {ranks[node]:.6f}")

PageRank вершины 1: 0.270327
PageRank вершины 2: 0.317529
PageRank вершины 3: 0.245014
PageRank вершины 4: 0.167130


In [7]:
import networkx as nx

G = nx.MultiDiGraph()
G.add_edges_from([(1, 2), (1, 2), (1, 3), (1, 3),
                  (2, 2), (2, 2), (2, 4), (2, 4),
                  (3, 1), (3, 3), (3, 5), (3, 4),
                  (4, 2), (4, 3), (4, 5), 
                  (5, 1), (5, 2), (5, 5),
                  ])
print("Out-degree:", dict(G.out_degree()))  # {1:1, 2:2, 3:0}
print("In-degree: ", dict(G.in_degree()))   # {1:0, 2:2, 3:1}

r = nx.degree_assortativity_coefficient(G, x='out', y='in')
print(f"Assortativity (out → in): {r:.4f}")

Out-degree: {1: 4, 2: 4, 3: 4, 4: 3, 5: 3}
In-degree:  {1: 2, 2: 6, 3: 4, 4: 3, 5: 3}
Assortativity (out → in): 0.0542


In [9]:
from collections import defaultdict
edges = [
    (1, 2), (1, 2),
    (1, 3), (1, 3),
    (2, 2), (2, 2),
    (2, 4), (2, 4),
    (3, 1), (3, 3), (3, 5), (3, 4),
    (4, 2), (4, 3), (4, 5),
    (5, 1), (5, 2), (5, 5),
]
in_deg = defaultdict(int)
out_deg = defaultdict(int)

for u, v in edges:
    out_deg[u] += 1
    in_deg[v] += 1

nodes = sorted(set(in_deg.keys()) | set(out_deg.keys()))
deg = {}
for n in nodes:
    deg[n] = in_deg[n] + out_deg[n]

print("Степени вершин:")
for n in nodes:
    print(f"deg({n}) = in({in_deg[n]}) + out({out_deg[n]}) = {deg[n]}")

Степени вершин:
deg(1) = in(2) + out(4) = 6
deg(2) = in(6) + out(4) = 10
deg(3) = in(4) + out(4) = 8
deg(4) = in(3) + out(3) = 6
deg(5) = in(3) + out(3) = 6


In [10]:
# Инициализация сумм для каждой вершины
sum_neighbor_deg = defaultdict(int)

for u, v in edges:
    # Ребро (u, v): u связан с v, v связан с u
    sum_neighbor_deg[u] += deg[v]
    sum_neighbor_deg[v] += deg[u]

# Теперь выведем суммы для вершин степени 6
target_nodes = [n for n in nodes if deg[n] == 6]
print("\nСуммы степеней соседей (с учётом кратности и петель):")
total_sum = 0
for n in target_nodes:
    s = sum_neighbor_deg[n]
    total_sum += s
    print(f"  Вершина {n}: сумма = {s}")

print(f"\nОбщая сумма по всем вершинам степени 6: {total_sum}")


Суммы степеней соседей (с учётом кратности и петель):
  Вершина 1: сумма = 50
  Вершина 4: сумма = 52
  Вершина 5: сумма = 42

Общая сумма по всем вершинам степени 6: 144
